# Hands-On Data Analysis with DuckDB

DuckDB is particularly well-suited to data analysis workflows due to its versatility and highly optimized performance, allowing practitioners to scale data analysis workflows beyond what they would be otherwise able to achieve on their local machine. Previously, we have been focusing more on covering core DuckDB concepts and features, with a bit of data analysis thrown in as examples. In this chapter, we'll be putting the data analysis workflow first by taking what we've learned about using DuckDB and using these foundations to perform some hands-on exploratory analysis of a dataset. This will allow us to explore different approaches for performing effective data analysis with DuckDB.

More specifically, in this chapter, we'll cover the following topics:
- Loading our dataset from a CSV file and applying some data cleaning steps, before writing it to a DuckDB database for our analysis
- Using the JupySQL library for convenient SQL queries in Jupyter Notebooks
- Using the Plotly data visualization library to create interactive
data visualizations
- Using DuckDB and Plotly together to perform exploratory
data analysis over a public dataset of pedestrian traffic counts
through Melbourne CBD

By the end of this chapter, you will be in a position to assemble workflows for performing effective data analysis with DuckDB that works for your particular needs.

## Technical requirements

### Setting up the environment

In order to run the examples in this notebook, you'll need to install the Python dependencies for this project. You can do this by running the following command in your terminal when in the root directory of the project. Note that ideally this should be using a Python virtual environment for this project.

In [ ]:
pip install -r requirements.txt

For complete instructions on how to set up your environment for working through the examples, please consult the *Technical requirements* section of the chapter *Setting up the DuckDB Python Client*.

### Obtaining the dataset

In this chapter, we'll be working with one of the pedestrian counting-system datasets that was produced and made public by the city of Melbourne, which contains hourly pedestrian counts from pedestrian sensors located in and around the Melbourne Central business district. We'll be working with a historical
snapshot of this dataset covering 2009 to 2022.

To download this snapshot, visit the dataset's home page (https://data.melbourne.vic.gov.au/explore/dataset/pedestrian-counting-system-monthly-counts-per-hour) and locate the ZIP file containing the 2009 to 2022 archive. Note that this download can be found as an attachment and is a different download from the current version of this dataset, which has a different schema from the archived snapshot that we'll be working with.

Once you've extracted the CSV file from the ZIP archive and placed it inside the chapter_11 directory, you're good to go. Note that the CSV file in the archive we'll be working with is named **Pedestrian_Counting_System_Monthly_counts_per_hour_may_2009_to_14_dec_2022.csv**. However, for the exercises in this chapter, we've renamed it to the shorter
**pedestrian_records_2009-2022.csv**.

In [ ]:
mkdir -p /workspaces/Getting-Started-with-DuckDB/chapter_11 && \
wget -O /workspaces/Getting-Started-with-DuckDB/chapter_11/pedestrian_counts.zip "https://data.melbourne.vic.gov.au/api/datasets/1.0/pedestrian-counting-system-monthly-counts-per-hour/attachments/pedestrian_counting_system_monthly_counts_per_hour_may_2009_to_14_dec_2022_csv_zip/" && \
unzip /workspaces/Getting-Started-with-DuckDB/chapter_11/pedestrian_counts.zip -d /workspaces/Getting-Started-with-DuckDB/chapter_11

## Preparing the pedestrian traffic dataset for analysis 

Before we can start analyzing the dataset, we need to load it into DuckDB in a shape that supports the kinds of analytical queries that will enable us to explore our dataset effectively. Our plan of attack will be to establish the steps required for parsing and transforming the CSV-backed dataset into a useful schema of columns and data types. After, we'll load this data into a table in a persistent on-disk DuckDB database, which will enable ongoing analysis across working sessions.

When working with DuckDB in Python, we have a choice between using the Relational API or the DB-API to work with DuckDB. The DB-API tends to be better suited when building applications as it promotes interoperability across data processing tools, whereas the Relational API offers a richer interface and features that are designed to enable effective data analysis; so, we will be working with the Relational API for data analysis.

### Establishing the data processing steps

The first step in our analysis will be to identify the data loading and transformation steps we'll use to get the data ready for performing our set of exploratory analyses. We'll do this first step using the DuckDB Python client's default database before we persist our transformation in a table in an on-disk database. The default database is the in-memory database that's automatically created when you import the duckdb module. This is used when you call methods associated with DuckDB connection objects against the duckdb module, such as **duckdb.sql()** and **duckdb.read_csv()**.

Let's start by reading the CSV using the **read_csv()** method. As this method is part of the Relational API, it will return a relation object, which we'll assign to a variable:

In [3]:
import duckdb

records = duckdb.read_csv("Pedestrian_Counting_System_Monthly_counts_per_hour_may_2009_to_14_dec_2022.csv") 

records.show(max_width=200) 

┌─────────┬───────────────────────────────┬───────┬──────────┬───────┬──────────┬───────┬───────────┬───────────────────────────────┬───────────────┐
│   ID    │           Date_Time           │ Year  │  Month   │ Mdate │   Day    │ Time  │ Sensor_ID │          Sensor_Name          │ Hourly_Counts │
│  int64  │            varchar            │ int64 │ varchar  │ int64 │ varchar  │ int64 │   int64   │            varchar            │     int64     │
├─────────┼───────────────────────────────┼───────┼──────────┼───────┼──────────┼───────┼───────────┼───────────────────────────────┼───────────────┤
│ 2887628 │ November 01, 2019 05:00:00 PM │  2019 │ November │     1 │ Friday   │    17 │        34 │ Flinders St-Spark La          │           300 │
│ 2887629 │ November 01, 2019 05:00:00 PM │  2019 │ November │     1 │ Friday   │    17 │        39 │ Alfred Place                  │           604 │
│ 2887630 │ November 01, 2019 05:00:00 PM │  2019 │ November │     1 │ Friday   │    17 │        37 

Note that we've explicitly called the relation's **show()** method, rather than relying on the notebook to call it implicitly for us. This is so that we can set **max_width** to a higher value, to prevent columns from being skipped and displayed.

As we can see, each record in this dataset contains the number of pedestrian counts passing through a given sensor for a specific hour, along with other information about the hourly reading, such as the sensor name and timestamp of the record, as well as date and time components extracted from the timestamp. Remember
that the result sets of relations are loaded lazily, so this only provides a preview of the first 10,000 records that have been loaded. Shortly, we'll see that there are more records than this in the dataset.

The primary data issue that you might have already noticed from this relation is that the **Date_Time** field is of the **VARCHAR** type as the CSV reader hasn't been able to automatically detect and parse the non-standard timestamp format of the field. For us to be able to perform some time series analysis and visualizations, we'll need to load this column as a **TIMESTAMP** type. Let's address this by providing appropriate parameters to the **read_csv()** method:

In [2]:
records = duckdb.read_csv( 
    "Pedestrian_Counting_System_Monthly_counts_per_hour_may_2009_to_14_dec_2022.csv", 
    dtype={"Date_Time": "TIMESTAMP"}, 
    timestamp_format="%B %d, %Y %H:%M:%S %p", 
)

NameError: name 'duckdb' is not defined

To parse the **Date_Time** field as **TIMESTAMP**, we needed to use the **dtype** parameter of the method to indicate the field's type and we also needed to use the **timestamp_format** parameter to instruct the CSV reader how to parse the string value into a timestamp. For the full reference on format specifiers that can be used when parsing dates and timestamps, see the DuckDB documentation: http://duckdb.org/docs/sql/functions/dateformat.

Let's check if our data cleaning has worked by looking at the first five records of the records relation:

In [6]:
records.limit(5).show(max_width=200)

┌─────────┬─────────────────────┬───────┬──────────┬───────┬─────────┬───────┬───────────┬──────────────────────────────┬───────────────┐
│   ID    │      Date_Time      │ Year  │  Month   │ Mdate │   Day   │ Time  │ Sensor_ID │         Sensor_Name          │ Hourly_Counts │
│  int64  │      timestamp      │ int64 │ varchar  │ int64 │ varchar │ int64 │   int64   │           varchar            │     int64     │
├─────────┼─────────────────────┼───────┼──────────┼───────┼─────────┼───────┼───────────┼──────────────────────────────┼───────────────┤
│ 2887628 │ 2019-11-01 17:00:00 │  2019 │ November │     1 │ Friday  │    17 │        34 │ Flinders St-Spark La         │           300 │
│ 2887629 │ 2019-11-01 17:00:00 │  2019 │ November │     1 │ Friday  │    17 │        39 │ Alfred Place                 │           604 │
│ 2887630 │ 2019-11-01 17:00:00 │  2019 │ November │     1 │ Friday  │    17 │        37 │ Lygon St (East)              │           216 │
│ 2887631 │ 2019-11-01 17:00:00 │ 

#### *USING **ENUM** TYPES TO OPTIMIZE THE DATA LOADING PROCESS*

*A further enhancement we could make to our data loading process would be to store the Month and Day columns as custom **ENUM** types. **ENUM** types provide a mapping from numerical values to the unique values seen in a column, with each value being encoded by the corresponding integer for its value. In our case, we could define two custom **ENUM** types: one for month names and one for the days of the week, using them as types for the Month and Day columns, respectively. This is particularly beneficial when we're dealing with string columns containing categorical values with low cardinality (that is, low numbers of distinct values), as is the case here, since the resulting **ENUM** columns only contain more efficient numerical values. This can result in a considerable reduction in storage space, as well as faster query petformance over these columns. For more information about creating and using **ENUM** types, consult the DuckDB documentation: https://duckdb.org/docs/sql/data_types/enum.html.*

Before loading our dataset into an on-disk database for analysis, we can also consider whether there are any data transformations we may want to apply. For example, we could drop any columns that we know we won't need. The additional date and time fields beyond the timestamp of the record are likely to be useful for us, and the **Sensor_ID** column can be used to link this dataset with a different dataset published by the city of Melbourne that contains information about each sensor, such as its geographic coordinates, so we should keep all these. The **ID** field, on the other hand, is not found in any other dataset concerning the pedestrian counting system, so we can drop this. Given that we're going to be performing a range of time series analyses, we'll almost certainly need these records to be ordered by timestamp, so the other transformation we'll perform involves sorting the records by the **Date_Time** column. Let's apply these transformations to our relation and inspect the result:

In [7]:
transformed = records.select("* EXCLUDE ID").sort("Date_Time")

Let's inspect the first five results from the **transformed** relation:

In [ ]:
transformed.limit(5).show(max_width=200)

### Loading the prepared dataset into a database table

Having established a process to ingest our CSV, we'll switch from using the default in-memory database to using a new on-disk database where we will load and persist our cleaned dataset as a table. Doing this gives us the advantage of only having to process our data once, so we won't have to wait for the CSV parsing to
happen every time we load our notebook. This is a particularly useful pattern for larger and more complex datasets, where the data loading process could be computationally intense and take some time. Another advantage of this approach is that it provides separation of concerns regarding data loading and data consumption, allowing multiple notebooks that perform different types of analyses to consume from the database without having to be concerned with data ingestion logic.

Given that we need to write to a specific database, we'll need to create a connection to a new on-disk database. Since we must close this connection after we're done writing our table to ensure our new database is updated safely, we'll use the new connection as a context manager. Inside the context manager's with **block**, we'll put all the data preparation steps, as well as create our new table, which we'll call **pedestrian_counts**. This will have the effect of automatically closing the connection after we finish writing to it:

In [8]:
with duckdb.connect("pedestrian.duckdb") as conn:
    result = (
        conn.read_csv(
            "Pedestrian_Counting_System_Monthly_counts_per_hour_may_2009_to_14_dec_2022.csv",
            dtype={"Date_Time": "TIMESTAMP"},
            timestamp_format="%B %d, %Y %H:%M:%S %p",
        )
        .select("* EXCLUDE ID")
        .sort("Date_Time")
    )
    result.to_table("pedestrian_counts")

## Effective data analysis using Jupyter Notebooks 

In this section, we're going to briefly cover two open source tools that will assist us in performing our data analysis within a Jupyter Notebook. The first is **JupySQL**, which provides us with a convenient way to run SQL queries in Jupyter notebooks. The second is **Plotly**, a comprehensive data visualization library that produces interactive visualizations with strong support for running inside Jupyter Notebooks. Let's get started.

### Convenient SQL queries with JupySQL

JupySQL is an open source Python package for streamlining the process of writing and running SQL queries in Jupyter Notebooks.

In [9]:
conn = duckdb.connect("pedestrian.duckdb")

conn.sql(
    """
    SELECT sum(Hourly_Counts) AS Total_counts
    FROM pedestrian_counts
    WHERE Year = 2022 AND Sensor_Name = 'Melbourne Central'
    """
)

┌──────────────┐
│ Total_counts │
│    int128    │
├──────────────┤
│      6897406 │
└──────────────┘

This approach to writing SQL becomes a bit cumbersome as it means having to do all your SQL query development inside a Python string. Using a triple-quoted string, as we did here, helps a little, but we still need to wrap every SQL query in the **sql()** method call, as well as the triple quotes, which will slow down our analysis feedback loop. It would be much more ergonomic if we could simply write the contents of our SQL string directly into a Jupyter Notebook cell and have the SQL query be extracted appropriately and automatically. This is exactly what JupySQL allows us to do.

To use JupySQL in a notebook, we need to configure JupySQL so that it knows which database to submit our queries to.

In [4]:
# configure JupySQL to use the default DuckDB database
%load_ext sql 
conn = duckdb.connect() 
%sql conn --alias duckdb 

The sql extension is already loaded. To reload it, use:
  %reload_ext sql


The commands prefixed with % characters are not Python code, but rather **magic commands** (or just magics), which are directives that are made available by the IPython kernel — the default Python kernel used by Jupyter. Magics enable you to interact with and configure the underlying kernel that runs your Python code inside the notebook. In the preceding example, **%load_ext** is a line-magic that loads the JupySQL extension into the notebook's kernel, at which point the **%sql** line-magic tells JupySQL to use the new in-memory database, conn, we created, while also giving it an alias of duckdb, which helps us keep track of which connection was used for each query in case we're working with multiple databases.

In our context, we want to work with the database we created in the previous section; so, let's configure JupySQL so that it uses a new connection to our on-disk **pedestrian.duckdb** database while also giving it an appropriate alias:

In [5]:
# for this exercise, we'll configure JupySQL to use a connection to the on-disk database
conn = duckdb.connect("pedestrian.duckdb")
%sql conn --alias pedestrian.duckdb

#### *ALTERNATIVE JUPYSQL-DUCKDB CONFIGURATION*
*In addition to using native DuckDB connections, there is an alternative way to connect JupySQL to DuckDB that involves connecting via the SQLAIchemy Python library. This enables you to use JupySQL's **%sqlcmd** line-magic, which is not supported with DuckDB native connections, and which allows you to gather a range of diagnostic information about tables in your database, as well as provide an interactive widget for creating and managing database connections. See the JupySQL docs for more details: https://jupysql.ploomber.io/ennatest/tutorials/duckdb-native-sqlalchemy.html.*

By default, JupySQL returns its own lazily-evaluated result-set object, which can then be operated on or converted into a dataframe. However, to conduct exploratory data analysis, rather than having to convert this into a dataframe each time, it would be more convenient to get back a dataframe directly. JupySQL supports conversion to both pandas and Polar dataframes, and for this analysis, we'll use pandas. So, let's configure JupySQL so that it automatically returns a pandas dataframe for each query:

In [6]:
# configure JupySQL to return Pandas dataframes by default
%config SqlMagic.autopandas = True 

An important caveat to be aware of is that by setting this parameter, we will lose the lazy evaluation that is associated with DuckDB's relation objects. All SQL commands that we run with JupySQL will have their query results fully materialized so that they can be converted into pandas dataframes. For our purposes, with the analyses we want to perform on this dataset, this will be fine; however, it is important to know that this does have performance implications, especially when working with large datasets.

With JupySQL configured, we can now run our previous SQL query much more simply through the use of JupySQL's **%%sql** cell-magic. Jupyter cell-magics provide instmctions to the IPython kernel to change how the contents of the cell are run. In this case, the %%sql cell-magic causes the contents of the cell to be submitted to the configured database as a SQL string for running.

Let's see this in action:

In [7]:
%%sql 
SELECT sum(Hourly_Counts) AS Total_Counts
FROM pedestrian_counts 
WHERE Year = 2022 AND Sensor_Name = 'Melbourne Central' 

Running query in 'pedestrian.duckdb'

,Total_Counts
0,6897406.0


Note that after running this cell, we haven't captured this dataframe as a variable; we've simply displayed it. We can access this resulting dataframe through the special **_** variable, which the IPython kernel automatically assigns the value of the final expression in the most recently executed cell. To illustrate this, we can use Python's built-in **type()** function:

In [14]:
type(_)

pandas.core.frame.DataFrame

JupySQL provides a convenient syntax for assigning the result of our query to a new variable. All we have to do is change our **%%sql** magic to **%%sql query_result <<**, which will assign the resulting dataframe to the **query_result** variable. Let's try this out with a different SQL query that calculates the total counts for each sensor in 2022 and sorts them in descending order:

In [8]:
%%sql sensors_2022_df << 
SELECT Sensor_Name,
    sum(Hourly_Counts)::BIGINT AS Total_Counts 
FROM pedestrian_counts 
WHERE Year = 2022 
GROUP BY Sensor_Name 
ORDER BY Total_Counts DESC 

Running query in 'pedestrian.duckdb'

Casting the result of sum to **BIGINT**, which pandas can convert into its **int64** data type, ensures that we wind up with integers in our dataframe. We can confirm that this has worked by inspecting the **dtypes** attribute of our dataframe:

In [16]:
sensors_2022_df.dtypes

Sensor_Name     object
Total_Counts     int64
dtype: object

We don't see any output after executing this cell as our resulting dataframe has been assigned to the **sensors_2022_df** variable.

Let's inspect the first 10 rows of this dataframe to find the top sensors for 2022 by total pedestrian counts:

In [17]:
sensors_2022_df.head(10)

,Sensor_Name,Total_Counts
0,Flinders La-Swanston St (West),10492872
1,Southbank,8737282
2,Melbourne Central,6897406
3,Elizabeth St - Flinders St (East) - New footpath,6511465
4,Princes Bridge,6202149
5,State Library - New,6049385
6,Flinders Street Station Underpass,5772514
7,Melbourne Convention Exhibition Centre,5634531
8,Bourke Street Mall (North),5614610
9,Melbourne Central-Elizabeth St (East),5380759


### Interactive visualisations with Plotly

Plotly is an open source Python package for producing data visualizations. Its ability to make interactive browser-based data visualizations makes it particularly useful in exploratory data analysis workflows, where interactive charts serve as valuable tools. This will be a very quick overview of working with Plotly so that we can use it within our data analysis. For a more comprehensive guide, see the Plotly Python client documentation: https://plotly.com/python.

The Plotly Python client is a wrapper around the underlying Plotly.js JavaScript library. When working with the Python client, we are essentially building chart configuration objects that are fed into Plotly.js for rendering in a browser. Plotly refers to these objects that encode our visualization's properties as figures. To make these figures, you can either build them up using lower-level
components of the **Plotly Python API** or you can use **Plotly Express**, a higher-level interface for rapidly creating Plotly Figure objects which is also included in the Plotly Python package. In general, a good heuristic is to use Plotly Express where possible, falling back to the lower-level interface only when you need more customization than is possible using Plotly Express alone. 

Plotly Express is primarily designed around creating visualizations from dataframes, supporting both pandas and Polars, with non-dataframe input also being supported. The workflow we'll use is to query DuckDB using JupySQL, and then feed the resulting dataframe into Plotly Express. The best way to work with Plotly Express is to import the **plotly.express** module under the more convenient name of **px**. Once we've done this, we can use it to call specific visualization functions that are needed for our analysis:

In [9]:
import plotly.express as px

By ending the cell with the **figure** variable that contains the **Figure** object that's returned from the **px.bar()** function, most Jupyter Notebook environments should automatically render this as a Plotly visualization. However, in some cases, you may need to explicitly call **figure.show()**. Once displayed, our figure will look like this:

In [10]:
figure = px.bar( 
    sensors_2022_df.head(10),
    x="Sensor_Name",
    y="Total_Counts",
    height=700,
    title="Top 10 sensors by traffic for 2022",
)

figure

#### *TITLE YOUR PLOTS!*

*As a matter of practice, it's a good habit to ensure all your visualizations have titles. While it may feel like a chore when you just want to get on with producing a plot, you will thank yourself upon coming back to your notebook later when your analysis is no longer fresh in your head. Importantly, this will also make your notebook a much more accessible artifact for anyone else working with or
reviewing your notebook.*

Let's make one more visualization with Plotly Express. This time, we'll visualize the number of active sensors across each year using a line plot. Plotly provides this functionality through the **px.line()** function. First, we'll need to query our database to get the distinct number of sensor names seen across each year, which we can do with the following DuckDB query:

In [11]:
%%sql sensor_years_df << 
SELECT Year, COUNT(DISTINCT Sensor_Name) AS Total_Sensors 
FROM pedestrian_counts 
GROUP BY Year 
ORDER BY Year 

Running query in 'pedestrian.duckdb'

In [12]:
sensor_years_df.head(5)

,Year,Total_Sensors
0,2009,18
1,2010,18
2,2011,18
3,2012,18
4,2013,32


To create a figure object that specifies our desired line plot, we'll call the **px.line()** function and pass it to our new dataframe, as well as specify the target columns for the *x* and *y* axes. Additionally, we'll use the marker parameter to configure the line plot so that it includes a marker for each data point, without which our plot would simply be a plain line:

In [13]:
figure = px.line( 
    sensor_years_df, 
    x="Year",
    y="Total_Sensors",
    markers=True,
    height=500,
    title="Total number of active sensors by year"
)

figure

Sometimes, you may want to customize your visualizations in ways that Plotly Express doesn't allow through its more concise interface. In these situations, we can fall back to Plotly's lower-level figure API by updating the properties of the figure object that's returned from Plotly Express functions. For example, in the chart shown above, we might want every year on the x-axis to have a label, rather than every other. We also might want to align the title so that it's in the center of the plot, rather than to the left. A general-purpose way to update layout properties such as these is to call the **Figure.update_figure()** method (for an existing Figure object) and pass its parameters with the new
properties.

In [14]:
figure.update_layout(xaxis={"dtick": 1}, title={"x": 0.5})

Plotly also provides a more convenient way of specifying nested properties through its "magic underscore" notation, which allows you to configure nested figure properties by joining property levels together using a _ character. Using this notation, we can rewrite the preceding line so that it's much simpler:

In [15]:
figure.update_layout(xaxis_dtick=1, title_x=0.5)

Two resources we'd recommend consulting if you want to further explore using Plotly in Python are as follows:
- The Plotly Express guide: https://plotly.com/python/plotly-express
- The Plotly Python Figure reference: https://plotly.com/python/reference/index

## Analysing pedestrian traffic through Melbourne CBD

Now that we've prepared our dataset and covered the tools we're going to use, let's jump into some analysis of the Melbourne pedestrian counting dataset. We'll continue to use DuckDB with JupySQL to query our pedestrian_counts table, and Plotly to make visualizations. Note that we won't always display the dataframes showing the results of our query.

### Visualizing total pedestrian counts over time

In [16]:
%%sql year_counts_df <<
SELECT Year, sum(Hourly_Counts)::BIGINT AS Total_Counts
FROM pedestrian_counts
GROUP BY Year
ORDER BY Year

Running query in 'pedestrian.duckdb'

Note that we didn't assign the result of the **px.line()** function to a variable as all we want to do is view the result. Since it's the only expression in this cell, our notebook will still render the resulting Figure object, which will look like this:

In [17]:
px.line(
    year_counts_df,
    x="Year",
    y="Total_Counts",
    markers=True,
    height=500,
    title="Yearly traffic across all sensors",
)

To address the confounding impact of the changing number of sensors over the years, let's filter down our dataset to only records from sensors that have readings across every year, before repeating the same analysis. We'll do this in two steps:
1. Extract the names of sensors that have readings for every year in the dataset, storing the result in a table called **common_sensors**.
2. Repeat our sum aggregation of pedestrian counts by year while also filtering our rows with sensor names that don't occur in the **common_sensors** table.

In [18]:
%sql SELECT count(DISTINCT Year) FROM pedestrian_counts

Running query in 'pedestrian.duckdb'

,"count(DISTINCT ""Year"")"
0,14


Ordinarily, when issuing a **CREATE TABLE** statement in DuckDB, no results are returned. When using JupySQL to submit database-modifying SQL, however, it returns the number of rows that were inserted into the database.

In [19]:
%%sql
CREATE OR REPLACE TABLE common_sensors AS
SELECT Sensor_Name
FROM pedestrian_counts
GROUP BY Sensor_Name
HAVING COUNT(DISTINCT Year) = 14;

Running query in 'pedestrian.duckdb'

,Count
0,15


In [20]:
%%sql year_counts_filtered_df <<
SELECT Year, sum(Hourly_Counts)::BIGINT AS Total_Counts
FROM pedestrian_counts
WHERE Sensor_Name IN (FROM common_sensors)
GROUP BY Year
ORDER BY Year

Running query in 'pedestrian.duckdb'

In [21]:
px.line(
    year_counts_filtered_df,
    x="Year",
    y="Total_Counts",
    markers=True,
    height=500,
    title="Yearly traffic for sensors active all years",
)

In [22]:
%%sql year_month_counts_df << 
SELECT
    Year,
    Month,
    month(Date_Time) AS Month_Num,
    sum(Hourly_Counts)::BIGINT AS Total_Counts
FROM pedestrian_counts
WHERE Year IN (2019, 2020, 2021)
    AND Sensor_Name in (FROM common_sensors)
GROUP BY Year, Month, Month_Num
ORDER BY Year, Month_Num

Running query in 'pedestrian.duckdb'

In [23]:
year_month_counts_df.head(15)

,Year,Month,Month_Num,Total_Counts
0,2019,January,1,7000243
1,2019,February,2,6079776
2,2019,March,3,7740551
3,2019,April,4,8281383
4,2019,May,5,7908534
5,2019,June,6,7323166
6,2019,July,7,7915558
7,2019,August,8,8013224
8,2019,September,9,7436127
9,2019,October,10,7665717


As you can see, this dataframe contains a record for each year-month pair, with its corresponding total counts, and is sorted by both year and count. To create our line plot, we can call **px.line()** with the **x** parameter set to Month, and the **y** parameter set to **Total_counts**. We also need to configure the resulting figure object so that it produces multiple lines, one for each year in our data extract. Plotly Express allows us to do this concisely by specifying that the **Year** column in our dataframe should be treated as a categorical variable whose distinct values are used to group the dataset into multiple lines, one for each year. Plotly refers to each data series occurring within a single figure as a **trace**. Plotly provides several ways to split an input dataframe into groups to create multiple traces that can be compared, each of which uses a different visual variable to encode each trace.

Since you may be viewing an electronic version of this book via a color display, or you could be reading a monochrome printed page, we'll use two visual variables to distinguish the different year plots: the color of each line and the shape of the line markers. To do this, we can pass **px.line()** the color and **symbol** keyword arguments, both with a value of Year. We'll also specify the specific marker shapes to use by passing the **symbol_squence** parameter as a list of symbol-identifier strings.

The other visual enhancement we made was to improve interpretability by setting the marker size that's used for each trace to a larger value by using the **update_traces()** method. When we evaluate the preceding cell so that we can render the resulting figure, we get the following visualization:

In [24]:
px.line(
    year_month_counts_df,
    x="Month",
    y="Total_Counts",
    color="Year",
    symbol="Year",
    symbol_sequence=["square", "diamond", "circle"],
    markers=True,
    height=500,
    title="Monthly traffic for sensors active 2019-2021",
).update_traces(marker_size=8)

### Time series plots of sensors

In [25]:
%%sql sensor_2020_df << 
SELECT Hourly_Counts, Date_Time 
FROM pedestrian_counts 
WHERE Sensor_Name = 'Flinders La-Swanston St (West)'
    AND Year = 2020

Running query in 'pedestrian.duckdb'

In [26]:
config = {
  'toImageButtonOptions': {
    'format': 'png',
    'filename': 'custom_image',
    'height': 500,
    'width': 1200,
    'scale':5
  }
}

This is a perfect time to take advantage of another of the interactive features that Plotly's visualizations offer. We can zoom in on specific regions of the x and y axes by left-clicking and dragging over the desired region. Selecting the region from January 19 to 31 gives us the following time series. To reset the zoom level, you can either double-click anywhere in the figure or click the home icon in the figure's **modebar**, which appears when hovering over Plotly figures.

In [29]:
px.line(
    sensor_2020_df,
    y="Hourly_Counts",
    x="Date_Time",
    height=500,
    title="Hourly traffic for Flinders La-Swanston St (West)",
)

It might be interesting to compare the daily traffic rhythms across several different sensors to see whether there are variations across different sensors. Let's do this for three sensors, just for September 2019.

In [30]:
%%sql multi_sensor_df <<
SELECT Sensor_Name, Hourly_Counts, Date_Time
FROM pedestrian_counts
WHERE Sensor_Name IN (
        'Flinders St-Spark La',
        'Bourke Street Mall (North)',
        'Southern Cross Station'
    )
    AND Year = 2019
    AND Month = 'September'

Running query in 'pedestrian.duckdb'

Now, we'll use the **px.line()** function to produce a facet plot, which contains multiple subplots. We can do this with either the **facet_col** or **facet_row** parameters, both of which take a column name and produce subfigures for each distinct value in the column. We'll use the **sensor_Name** column to produce time series subfigures for each sensor. To support comparison across sensors, these subfigures would be best arranged as horizontal rows on top of each other. We can achieve this by setting the **facet_row** parameter to **sensor_Name**.

The one layout change we made to the entire figure was to the **fixedrange** property of the y-axis, which we set to **True**. This has the effect of locking the y-axis's range when zooming so that we can only change the x-axis range, which will make interactive exploration much easier.

In [31]:
px.line(
    multi_sensor_df,
    y="Hourly_Counts",
    x="Date_Time",
    facet_col="Sensor_Name",
    facet_col_wrap=1,
    title="Hourly pedestrian traffic for December 2019",
    height=800,
).update_layout(yaxis_fixedrange=True)

## Visualising the distribution of hourly pedestrian traffic

Time series plots are good at identifying broad seasonal and daily trends and are also useful for spotting anomalies. However, they are limited to plotting individual data points in time. If we want to draw some deeper insights into how pedestrian traffic varies across central Melbourne (and over time), we'll need to move away from visualizing individual data points to visualizing aggregated statistics across multiple data points. A frequently used plot for visualizing the distribution of a collection of values is a **box plot**.

This plot provides a graphical representation of which value the dataset centers around, how spread out the values of the dataset are, and whether the dataset is skewed toward a particular end of the distribution. If you haven't worked with them before, you may find the following Wikipedia article on box plots a helpful
reference as you work through this section: https://en.wikipedia.org/wiki/Box_plot.

In [32]:
%%sql bourke_daily_df << 
SELECT
    Year,
    Date_Time::DATE AS Date,
    sum(Hourly_Counts)::BIGINT AS Daily_Counts,
FROM pedestrian_counts
WHERE Sensor_Name = 'Bourke Street Mall (North)'
    AND Year IN (2019, 2020, 2021)
GROUP BY Year, Date

Running query in 'pedestrian.duckdb'

In [33]:
bourke_daily_df.head()

,Year,Date,Daily_Counts
0,2019,2019-03-01,35010
1,2019,2019-03-02,31290
2,2019,2019-03-03,20491
3,2019,2019-03-04,30589
4,2019,2019-03-05,27503


In [34]:
px.box(
    bourke_daily_df,
    x="Year",
    y="Daily_Counts",
    points="all",
    height=600,
    title="Distributions of daily traffic for a sensor",
)

These three box plots provide us with a rich set of information about the distribution of daily pedestrian counts across the three years. Hovering over each box plot informs us of the value of key summary statistics, including that they include the median daily count (the horizontal line inside the box), the first and third quartiles (the bottom and top of the boxes, enclosing 50% of the data points), and the lower and upper fence values (the horizontal lines bounding the whiskers extending from the box), which indicate sentinel data points, below or above which any data points are considered outliers. For the box plots of these three years, we can't see any outlier data points occurring outside the whiskers.

Now that you have all the ingredients you need, we'll hand over the keyboard to you so that you can continue the analysis. This would also be a good time to
look for opportunities to get some hands-on practice with other DuckDB SQL features that we covered in previous chapters, as well as other features from DuckDB's rich SQL API that we didn't have room to cover in this book.

## Summary

It's worth reflecting on how we interacted with DuckDB throughout this journey. When we prepared and loaded our dataset into a DuckDB table, we composed the necessary query using the DuckDB Python client's Relational API. When we moved on to querying this table for our data analysis, we switched to writing our SQL queries directly. JupySQL took care of dispatching these to our DuckDB database connection. There is no "right" way to arrive at this decision; we could have used either approach exclusively, or we could have even mixed them up. The best choice is the one that will make you and your team the most effective. Some factors to consider are that, the composability of the Relational API via Python expressions often makes it a better choice for building out data applications using DuckDB. On the other hand, some more complex analytics queries may be easier to work with in SQL, or sometimes, this may also be necessary, given that the Python Relational API does not (at the time of writing) offer full support for the DuckDB SQL API.

Having worked through this chapter, you've now seen some tangible real-world examples of how DuckDB can be used to drive effective exploratory data analysis. The tools and workflows that we put into action are ones that you can use and incorporate into your data analysis toolkit.